## 데이터 점검
    - 각 파일의 행 수, 컬럼별 dtype
    - 숫자여야 하는데 문자열로 읽힌 컬럼
    - 컬럼별 결측 수와 비율
    - bike_id, rental_id 의 중복 건수
    - bike_type, district, payment_method의 고유값 목록

In [ ]:
import pandas as pd

# 출력 시 자동 줄바꿈을 위한 pprint
from pprint import pprint
from config import raw_bikes_path, raw_rentals_path, stations_path, ENCODING

raw_bikes = pd.read_csv(raw_bikes_path(), encoding=ENCODING)
raw_rentals = pd.read_csv(raw_rentals_path(), encoding=ENCODING)
stations = pd.read_csv(stations_path(), encoding=ENCODING)

In [13]:
# 각 파일의 행 수, 컬럼별 dtype
def row_count_dtype(df):
    """
        데이터프레임의 행 수, 컬럼별 dtype 딕셔너리를 튜플 형태로 반환
    """
    dtypes = {col_name:col_dtype.name for col_name, col_dtype in df.dtypes.items()}
    return len(df),dtypes

pprint(f"raw_bikes : {row_count_dtype(raw_bikes)[0]:,}행, {row_count_dtype(raw_bikes)[1]}")
pprint(f"raw_rentals: {row_count_dtype(raw_rentals)[0]:,}행, {row_count_dtype(raw_rentals)[1]}")
pprint(f"stations: {row_count_dtype(stations)[0]:,}행, {row_count_dtype(stations)[1]}")

("raw_bikes : 55행, {'bike_id': 'str', 'station_id': 'str', 'bike_type': 'str', "
 "'gear_count': 'str', 'manufacture_year': 'str', 'daily_fee': 'str'}")
("raw_rentals: 15,000행, {'rental_id': 'str', 'bike_id': 'str', 'user_id': "
 "'str', 'rent_time': 'str', 'return_time': 'str', 'distance_km': 'float64', "
 "'fee': 'str', 'payment_method': 'str'}")
"stations: 30행, {'station_id': 'str', 'station_name': 'str', 'district': 'str'}"


In [17]:
# 숫자여야 하는데 문자열로 읽힌 컬럼
BIKES_NUM_COLS = ["gear_count", "daily_fee", "manufacture_year"]
RENTALS_NUM_COLS = ["distance_km", "fee"]

def num_cols_check(num_cols, df:pd.DataFrame):
    for col in num_cols:
        if not any(d in df[col].dtype.name for d in ["int", "float"]):
            print(f"{col}: {df[col].dtype}")

print("* raw-bikes")
num_cols_check(BIKES_NUM_COLS, raw_bikes)
print()
print("* raw-rentals")
num_cols_check(RENTALS_NUM_COLS, raw_rentals)

* raw-bikes
gear_count: str
daily_fee: str
manufacture_year: str

* raw-rentals
fee: str


In [23]:
# 컬럼별 결측 수와 비율
def nan_count_ratio(df:pd.DataFrame):
    nan_counts = df.isna().sum()
    nan_ratios = (nan_counts * 100 / len(df)).round(2)
    result = pd.DataFrame({
        "결측 수": nan_counts,
        "결측 비율(%)": nan_ratios
    })

    result = result[result["결측 수"] > 0]
    return result

print(f"raw_bikes  \n {nan_count_ratio(raw_bikes)}")
print()
print(f"raw_rentals\n {nan_count_ratio(raw_rentals)}")

raw_bikes  
                   결측 수  결측 비율(%)
station_id           1      1.82
manufacture_year     6     10.91

raw_rentals
              결측 수  결측 비율(%)
distance_km   212      1.41


In [20]:
# bike_id, rental_id 의 중복 건수
def dup_check(df:pd.DataFrame, subset):
    dup_count = df.duplicated(subset).sum()
    return dup_count

print(f"bike_id 중복체크: {dup_check(raw_bikes, ['bike_id'])}행")
print(f"rental_id 중복체크: {dup_check(raw_rentals, ['rental_id'])}행")

bike_id 중복체크: 5행
rental_id 중복체크: 500행


In [26]:
# bike_type, district, payment_method의 고유값 목록
bike_nuq, bike_uq = raw_bikes["bike_type"].nunique(), raw_bikes["bike_type"].unique()
district_nuq, district_uq = stations["district"].nunique(), stations["district"].unique()
payment_nuq, payment_uq = raw_rentals["payment_method"].nunique(), raw_rentals["payment_method"].unique()

print(bike_nuq, bike_uq)
print(district_nuq, district_uq)
print(payment_nuq, payment_uq)

print()
raw_rentals2 = pd.read_csv(raw_rentals_path(), encoding=ENCODING,
                          keep_default_na=False, dtype=str)
pprint(f"{raw_rentals2['distance_km'].unique().tolist()}")

6 <StringArray>
['일반　', '일 반', '일반', '전동', 'electric', '일반형']
Length: 6, dtype: str
10 <StringArray>
['강남구', '강동구', '강북구', '강서구', '관악구', '광진구', '구로구', '금천구', '노원구', '도봉구']
Length: 10, dtype: str
13 <StringArray>
['CARD', ' APP', 'membership', 'Membership', 'APP ', 'app', 'CARD ', 'MEMBERSHIP', 'App', ' CARD', 'card', 'Card',
 'APP']
Length: 13, dtype: str

("['13.1', '13.8', '2.0', '8.9', '10.1', '14.3', '14.5', '10.5', '3.6', "
 "'13.5', '5.2', '7.9', '12.1', '9.9', '6.7', '14.6', '2.4', '1.5', '3.8', "
 "'11.2', '4.9', '2.5', '5.0', '10.9', '7.1', '1.7', '3.5', '6.2', '8.7', "
 "'4.8', '0.9', '12.5', '9.5', '1.4', '6.1', '4.2', '3.0', '12.7', '13.4', "
 "'4.3', '13.3', '10.8', '9.1', '4.1', '4.4', '11.0', '5.7', '1.0', '10.7', "
 "'12.3', '1.2', '12.2', '9.8', '12.8', '1.6', '3.3', '6.5', '1.1', '7.2', "
 "'10.2', '9.2', '9.6', '7.6', '11.6', '9.4', '6.9', '7.3', '11.4', '7.5', "
 "'6.8', '7.7', '11.9', '11.1', '11.8', '10.0', '9.7', '13.0', '10.4', '13.6', "
 "'2.2', '5.5', '8.1', '

## 확인 문항
1. raw-bikes.csv는 55행인데 자전거는 몇 대입니까? 왜 다릅니까?
> 50대로 추정됨. (다른 이유: bike_id의 중복 건수가 5)
2. bike_type의 고유값은 몇 종류입니까? 실제로는 몇 종류여야 합니까?
> 6종으로 확인되나 실제로는 2종(일반/전동)이어야 함.
3. 숫자로 바꿀 수 없는 distance_km 값에는 어떤 것들이 있습니까?
> 'N/A'